# FASE 2. LIMPIEZA DE LOS DATOS - RUIDO (sonometros)

## 1. Carga de datos originales

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("TFM_Ruido_Valencia") \
    .getOrCreate()

# Ruta del archivo unificado (son los datos originales solo que el campo intensity tiene el mismo tipo de datos en todos los meses)
ruta_ruido = "gs://tfm_unir1/ruido/ruido_limpio"

# Cargar los datos
spark.read.parquet(ruta_ruido).createOrReplaceTempView("ruido_fase1")

spark.sql("""
    SELECT COUNT(*)                  AS registros,
           COUNT(DISTINCT id_sensor) AS sensores
    FROM ruido_fase1
""").toPandas()

26/11/24 20:21:57 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


,registros,sensores
0,41705447,141


## 2. Selección de columnas de interés 

En la fase 1 ya se seleccionaron las columnas necesarias: red, id_sensor, laeq y fecha_hora

## 3. Separación de los datos según su frecuencia temporal

In [3]:
spark.sql("""
    SELECT red,
           COUNT(DISTINCT id_sensor)                       AS sensores,
           COUNT(*)                                        AS registros,
           ROUND(COUNT(*) / COUNT(DISTINCT id_sensor), 1)  AS mediciones_por_sensor
    FROM ruido_fase1
    WHERE fecha_hora >= '2025-06-10 10:00:00'
      AND fecha_hora <  '2025-06-10 11:00:00'
    GROUP BY red
    ORDER BY red
""").toPandas()

,red,sensores,registros,mediciones_por_sensor
0,dipuval,31,86,2.8
1,hopu,20,1211,60.6
2,ruzafa,16,960,60.0
3,zas,15,900,60.0


Cada red mide con una frecuencia distinta: minutos, 15 minutos, etc. Para poder compararlas y cruzarlas, se agrupan en un valor por sensor y por hora

## 4. Limpieza de los datos 

### Datos con entities válidas

In [6]:
spark.read.csv("gs://tfm_unir1/ruido/entities.csv", header=True).createOrReplaceTempView("sensores")

# Nos quedamos solo con los sensores que tienen ficha (y coordenadas) en entities.csv
spark.sql("""
    CREATE OR REPLACE TEMP VIEW ruido_con_ficha AS
    SELECT r.*
    FROM ruido_fase1 r
    JOIN sensores s
      ON r.id_sensor = s.entityID
""")

spark.sql("""
    SELECT COUNT(*)                  AS registros,
           COUNT(DISTINCT id_sensor) AS sensores
    FROM ruido_con_ficha
""").toPandas()

,registros,sensores
0,41232832,137


### Agrupamos por sensor y por hora

In [10]:
# Un valor por sensor y hora. Los dB se promedian en energía porque la escala es logarítmica.
# Se quitan las horas con menos de 30 mediciones en las redes que miden cada minuto.
# En DipuVal se conservan todas (cada registro ya es un valor de 15 minutos).
spark.sql("""
    CREATE OR REPLACE TEMP VIEW ruido_horario AS
    SELECT red,
           id_sensor,
           DATE_TRUNC('hour', fecha_hora)                  AS fecha_inicio,
           ROUND(10 * LOG10(AVG(POWER(10, laeq / 10))), 1) AS laeq,
           COUNT(*)                                        AS mediciones
    FROM ruido_con_ficha
    GROUP BY red, id_sensor, DATE_TRUNC('hour', fecha_hora)
    HAVING red = 'dipuval' OR COUNT(*) >= 30
""")

spark.sql("""
    SELECT red, COUNT(*) AS horas
    FROM ruido_horario
    GROUP BY red
    ORDER BY red
""").toPandas()

,red,horas
0,dipuval,227006
1,hopu,227813
2,ruzafa,237004
3,zas,213341


Los niveles en dB se promedian en energía, porque la escala es logarítmica

## 5. Estandarización de variables 

### Verificación de la zona horaria

In [11]:
por_hora = spark.sql("""
    SELECT HOUR(fecha_inicio) AS hora,
           red,
           ROUND(AVG(laeq), 1) AS laeq_medio
    FROM ruido_horario
    GROUP BY HOUR(fecha_inicio), red
""").toPandas()

por_hora.pivot(index="hora", columns="red", values="laeq_medio")

red,dipuval,hopu,ruzafa,zas
hora,,,,
0,56.3,61.7,54.8,59.0
1,55.7,60.9,52.4,56.0
2,55.4,58.8,49.7,53.7
3,55.8,57.7,50.2,52.0
4,57.2,58.8,53.2,50.3
5,60.4,60.1,56.2,52.3
6,61.7,61.5,58.8,55.4
7,61.9,62.1,59.8,59.2
8,62.0,62.5,60.0,61.5


Los archivos de DipuVal, HOPU y Ruzafa indican que sus fechas están en UTC; los de ZAS no indican zona horaria. Al comparar el ruido medio por hora, ZAS tiene su mínimo de madrugada y su subida de la mañana unas 2 horas más tarde que las demás redes, lo que indica que sus fechas ya están en hora local. Por eso se convierten a hora de Valencia solo las tres primeras.

In [12]:
# DipuVal, HOPU y Ruzafa vienen en UTC: se pasan a hora de Valencia.
# ZAS ya viene en hora local: se deja igual.
spark.sql("""
    CREATE OR REPLACE TEMP VIEW ruido_local AS
    SELECT red,
           id_sensor,
           CASE WHEN red = 'zas' THEN fecha_inicio
                ELSE FROM_UTC_TIMESTAMP(fecha_inicio, 'Europe/Madrid')
           END AS fecha_inicio,
           laeq,
           mediciones
    FROM ruido_horario
""")

DataFrame[]

In [13]:
hora_local= spark.sql("""
    SELECT HOUR(fecha_inicio) AS hora,
           red,
           ROUND(AVG(laeq), 1) AS laeq_medio
    FROM ruido_local
    GROUP BY HOUR(fecha_inicio), red
""").toPandas()

hora_local.pivot(index="hora", columns="red", values="laeq_medio")

red,dipuval,hopu,ruzafa,zas
hora,,,,
0,57.7,63.7,58.1,59.0
1,56.7,62.5,56.0,56.0
2,56.1,61.4,54.3,53.7
3,55.5,60.7,51.6,52.0
4,55.4,58.0,48.6,50.3
5,56.1,57.5,51.1,52.3
6,57.7,59.3,54.1,55.4
7,61.4,60.5,57.5,59.2
8,61.8,62.0,59.7,61.5


Tras la conversión, las cuatro redes tienen su mínimo entre las 4 y las 5 h y empiezan a subir entre las 6 y las 7 h, lo que confirma que todas quedan en hora de Valencia.

In [14]:
# Agregamos variables de fecha con los mismos nombres que en trafico
spark.sql("""
    CREATE OR REPLACE TEMP VIEW ruido_silver AS
    SELECT red,
           id_sensor,
           fecha_inicio,
           laeq,
           mediciones,
           YEAR(fecha_inicio)       AS anio,
           MONTH(fecha_inicio)      AS mes,
           DAY(fecha_inicio)        AS dia,
           HOUR(fecha_inicio)       AS hora,
           DAYOFWEEK(fecha_inicio)  AS dia_semana
    FROM ruido_local
    WHERE YEAR(fecha_inicio) IN (2024, 2025)
""")

spark.sql("""
    SELECT red, id_sensor, CAST(fecha_inicio AS STRING) AS fecha_inicio,
           laeq, anio, mes, dia, hora, dia_semana
    FROM ruido_silver
    LIMIT 5
""").toPandas()

,red,id_sensor,fecha_inicio,laeq,anio,mes,dia,hora,dia_semana
0,hopu,NoiseLevelObserved-HOPVLCi6,2024-11-12 00:00:00,79.4,2024,11,12,0,3
1,hopu,NoiseLevelObserved-HOPVLCi6,2024-11-17 11:00:00,66.5,2024,11,17,11,1
2,hopu,NoiseLevelObserved-HOPVLCi15,2024-11-18 16:00:00,63.0,2024,11,18,16,2
3,hopu,NoiseLevelObserved-HOPVLCi15,2024-11-19 17:00:00,61.3,2024,11,19,17,3
4,hopu,NoiseLevelObserved-HOPVLCi11,2024-11-21 16:00:00,64.6,2024,11,21,16,5


## 7. Validación final de los datos preparados

In [15]:
spark.sql("""
    SELECT (SELECT COUNT(*) FROM ruido_fase1)  AS registros_fase1,
           (SELECT COUNT(*) FROM ruido_silver) AS registros_horarios,
           (SELECT COUNT(DISTINCT id_sensor) FROM ruido_silver) AS sensores
""").toPandas()

,registros_fase1,registros_horarios,sensores
0,41705447,905090,137


**Conclusiones:**
- La limpieza reduce el conjunto de 41.705.447 mediciones a 905.090 registros horarios (−97,8 %). 
- La mayor parte de la reducción corresponde a la agrupacion de mas mediciones minutales en un valor por sensor y hora. El resto corresponde a lagunos sensores sin ficha, a horas con menos de 30 mediciones y a las ultimas horas del 2025 que pasan al 2026.
- Todas las fechas quedan en hora de Valencia

## 8. Exportación de los datasets limpios

In [16]:
ruta_silver = "gs://tfm_unir1/processed/ruido/sonometros/"

spark.table("ruido_silver").write \
    .mode("overwrite") \
    .partitionBy("anio", "mes") \
    .parquet(ruta_silver)

print("Datos limpios guardados correctamente")

Datos limpios guardados correctamente
Datos limpios guardados correctamente


In [17]:
# Verificacion final
spark.read.parquet(ruta_silver).createOrReplaceTempView("ruido_verificacion")

spark.sql("""
    SELECT red,
           COUNT(*)                  AS registros,
           COUNT(DISTINCT id_sensor) AS sensores,
           CAST(MIN(fecha_inicio) AS STRING) AS desde,
           CAST(MAX(fecha_inicio) AS STRING) AS hasta,
           MIN(laeq) AS minimo,
           MAX(laeq) AS maximo
    FROM ruido_verificacion
    GROUP BY red
    ORDER BY red
""").toPandas()


,red,registros,sensores,desde,hasta,minimo,maximo
0,dipuval,226961,84,2025-02-11 10:00:00,2025-12-31 23:00:00,49.0,108.9
1,hopu,227801,20,2024-04-17 18:00:00,2025-12-31 23:00:00,39.3,106.8
2,ruzafa,236987,17,2024-04-18 08:00:00,2025-12-31 23:00:00,27.3,102.6
3,zas,213341,16,2024-01-08 08:00:00,2025-12-22 11:00:00,30.3,105.3


In [ ]:
## 9. Exportación de los datos de sonometros - distritos de la bronze a la silver en formato Parquet

In [6]:
# Tabla sonometro-distrito generada con QGIS (bronze)
spark.read.csv("gs://tfm_unir1/sonometros_distritos.csv", header=True, inferSchema=True) \
     .write.mode("overwrite") \
     .parquet("gs://tfm_unir1/processed/sonometros_distritos/")

# Verificación
spark.read.parquet("gs://tfm_unir1/processed/sonometros_distritos").createOrReplaceTempView("sonometros_distritos")
spark.sql("SELECT COUNT(*) AS sensores, COUNT(DISTINCT coddistrit) AS distritos FROM sonometros_distritos").toPandas()

,sensores,distritos
0,143,16
